# Reproducing Brudner, Zhou et al. (2026) from dandiset 001871

**Paper.** Brudner S, Zhou B, Jayaram V, Madeira Santana G, Crimaldi J, Clark DA, Emonet T (2026).
Fly navigational responses exploit plume-specific odor motion and gradient cues. *PNAS* 123(29)
e2608896123. [doi:10.1073/pnas.2608896123](https://doi.org/10.1073/pnas.2608896123). The claims file
(`claims/brudner-2026.yaml`) was written from the version of record and its SI Appendix; this attempt
read the same text, under `paper/`.

**Dandiset.** [DANDI:001871](https://dandiarchive.org/dandiset/001871/0.260630.1657), version
0.260630.1657. The paper cites no version. This is the only published version and the one the paper's
code (github.com/emonetlab/GradientMotionMultiplePlumes, commit effc7b0, `metadata/published_panel_params.json`)
names as the source of its Fig. 5 tables. The draft has the same 136 assets and total size.

**What the dandiset holds.** 131 behavior NWB files, one per trial of a fly cohort (83 smooth-plume
files, "nagel-smoke"; 24 "smoke-1a" and 24 "smoke-2a" complex-plume files), with per-trajectory
position, heading, velocities and the odor intensity, gradient and motion signals at the antennae,
and a table of turn events; the complex plume movie window (frames 300 to 1799 of smoke_1a_orig);
three freestream plume windows; and a compact asset of Fig. S1 panel inputs. The smooth plume movie
is not in it: it is on Dryad (doi:10.5061/dryad.g27mq71), which answers 401 to downloads without an
API token (printed in the data check), and it was not staged for this run, so every claim that needs it is blocked. The paper's
Yale Dataverse deposit (doi:10.60600/YU/590UBV) holds the behavior tables only.

**Outside inputs.** No data. Method details come from the paper's code at the commit above (cited in
`plumes.py`), and its published Fig. 5 tables (the claims file's expected values) are read from the
same commit only to show which files and readings they come from.

| Claim | Tested here |
|---|---|
| fig5e-cue-betas | tested |
| fig5f-cue-dominance | tested |
| figs5c-turn-angles | tested |
| fig2c-complex-auc | tested |
| fig2e-complex-weights | tested |
| figs1fg-resolution-robustness | tested |
| methods-intensity-right-bias | blocked, metadata_insufficient (the direction code has no left/right meaning); the result under each mapping is shown after fig5e |
| figs2c-one-frame-delay | blocked (smooth plume not readable); the complex part is shown |
| figs1-antisymmetry-index, figs1-temporal-statistics | blocked (smooth plume not readable); the freestream and complex parts are shown |
| fig1eh, fig1il, fig2b, fig2d | blocked (smooth plume not readable) |
| fig3bc, fig3d, fig3e, fig3ij, fig4ef, discussion-ten-percent-gain, figs4, figs5fh | blocked (smooth plume not readable; networks and agent simulations depend on it) |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

import plumes

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = plumes.list_assets()
assets["plume"] = assets.path.map(plumes.plume_of)
behavior_assets = assets[assets.plume.notna()]
sessions = plumes.load_behavior(behavior_assets, workers=8)
print(f"dandiset {plumes.DANDISET_ID} version {plumes.DANDISET_VERSION}: {len(assets)} assets; "
      f"loaded {len(sessions)} behavior files in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
kind = np.where(assets.plume.notna(), "behavior " + assets.plume.fillna(""), assets.path.str.split("/").str[0])
print("Assets by kind (count, GB):")
print(assets.assign(kind=kind).groupby("kind")["size"].agg(["count", "sum"]).assign(sum=lambda d: (d["sum"] / 1e9).round(2)).to_string())

info = pd.DataFrame([s["info"] for s in sessions])
info["trajectories"] = [s["frames"].trj.nunique() for s in sessions]
info["frames"] = [len(s["frames"]) for s in sessions]
info["turns"] = [len(s["turns"]) for s in sessions]
info["date"] = info.start.str[:10]
print("\nBehavior files by plume group:")
print(info.groupby("plume")[["trajectories", "frames", "turns"]].agg(["count", "sum"]).to_string())
print("\nFiles by recording date and group:")
print(pd.crosstab(info.date, info.plume).to_string())
print("\nDistinct session ids:", info.session_id.nunique(), "; distinct subject ids:", info.subject_id.nunique())
print("Series timing (rate or timestamps):", info.rates.astype(str).value_counts().to_dict())
print("Units stated:", sessions[0]["info"]["units"], "; position:", sessions[0]["info"]["xy_unit"])
desc = info.description.str.replace("Olfactory navigation behavior session: ", "")
info["stem"] = desc.str.extract(r"^(nagel_smoke|smoke_1a|smoke_2a)")[0]
info["vflip"] = np.where(desc.str.contains("vflip1|vflipTrue"), "flipped", "not flipped")
print("Session descriptions: movie stem and projection flip by group:")
print(pd.crosstab([info.plume, info.stem], info.vflip).to_string())
print("Example session descriptions:", desc.groupby(info.plume).first().to_dict())

In [ ]:
# The smooth plume movie is on Dryad (doi:10.5061/dryad.g27mq71): its file list, and the answer to a download without a token
import requests
DRYAD = "https://datadryad.org"
dv = requests.get(DRYAD + "/api/v2/datasets/doi%3A10.5061%2Fdryad.g27mq71", timeout=60).json()["_links"]["stash:version"]["href"]
for f in requests.get(DRYAD + dv + "/files", timeout=60).json()["_embedded"]["stash:files"]:
    if f["path"].endswith(".h5"):
        r = requests.get(DRYAD + f["_links"]["stash:download"]["href"], timeout=60, allow_redirects=False, stream=True)
        print(f"Dryad {dv}: {f['path']}, {f['size'] / 1e9:.2f} GB; download without a token answers HTTP {r.status_code}")
        r.close()

The paper reports no count of cohorts, flies, trajectories or turns. Each file is one 2-minute trial
(session descriptions name the movie, the vial, the flip of the projection and a trial index);
trajectories are fly tracks, not flies, and a fly cannot be followed across trajectories, so the unit of
clustering is the trajectory, as in the paper (SI, Behavioral Analysis). The two complex groups were
recorded on the same days and vials, four files per vial and day for each, and their session
descriptions name two different movies, "smoke_1a" and "smoke_2a", each with 12 flipped and 12 unflipped
trials. The dandiset's description says the complex plume movie it deposits, smoke_1a_orig, "was
analyzed throughout the paper".

In [ ]:
frames_all = pd.concat([s["frames"].assign(plume=s["info"]["plume"]) for s in sessions], ignore_index=True)
cols = ["x", "y", "theta", "dtheta", "vx", "spd", "signal", "spatial_gradient", "odor_velocity"]
print("Ranges of every variable used, by group (1st, 50th, 99th percentile, min, max, NaN fraction):")
rng_tab = frames_all.groupby("plume")[cols].agg([lambda v: np.nanpercentile(v, 1), "median", lambda v: np.nanpercentile(v, 99), "min", "max", lambda v: v.isna().mean()])
rng_tab.columns = [f"{c}|{s}" for c, s in zip(rng_tab.columns.get_level_values(0), ["p1", "p50", "p99", "min", "max", "nan"] * len(cols))]
print(rng_tab.T.round(3).to_string())
far = frames_all.y > 190
print("\nArena (Methods): 270 x 170 mm. Frames with y > 190 mm:", int(far.sum()),
      "; their y:", round(frames_all.y[far].min(), 1), "to", round(frames_all.y[far].max(), 1), "mm, in group(s)", frames_all.loc[far, "plume"].unique().tolist())
print("  files:", frames_all.loc[frames_all.y > 190].groupby("session").size().to_dict())

# all-NaN trajectories and repeated recordings
traj = frames_all.groupby(["plume", "session", "trj"]).agg(n=("x", "size"), nan=("x", lambda v: v.isna().mean()))
print("\nTrajectories with no position at all:", int((traj.nan == 1).sum()), "of", len(traj))
empty = set(traj.index[traj.nan == 1].droplevel(0))
n_on_empty = sum(int(s["turns"].apply(lambda r: (s["info"]["session_id"], r.trj) in empty, axis=1).sum()) for s in sessions)
print("Turn events on those trajectories:", n_on_empty)
import hashlib
hashes = {}
for (pl, se, tr), g in frames_all.groupby(["plume", "session", "trj"]):
    if g.x.notna().any():
        hashes.setdefault(hashlib.md5(np.round(g[["x", "y", "theta"]].to_numpy(), 4).tobytes()).hexdigest(), []).append((se, tr))
dups = [v for v in hashes.values() if len(v) > 1]
print("Trajectories with position whose x, y and heading repeat another trajectory exactly:", len(dups))
th = {}
for s in sessions:
    th.setdefault(hashlib.md5(np.round(s["turns"][["start_time", "turn_x", "turn_y"]].to_numpy(), 4).tobytes()).hexdigest(), []).append(s["info"]["session_id"])
print("Turn tables repeated across files:", [v for v in th.values() if len(v) > 1])
ids = frames_all[["session", "trj"]].drop_duplicates()
print("Trajectory ids unique across files:", ids.trj.is_unique, "(clusters are named by file and id)")

No recording is deposited twice: no trajectory with position repeats another, and no turn table
repeats. 205 trajectories have no position at all (they are left out of the comparison of repeats, as
any two all-NaN trajectories of equal length look identical); the number of turn events on them is
printed above. Four smooth files have 10 frames with positions far outside the 170 mm
arena width (y of 768 to 814 mm), an error of the tracking or the conversion; these frames fall outside the analysis region of
Fig. 5 (y 67 to 97 mm) and do not enter any test. `odor_velocity` (motion) is defined on only about 3%
of frames; the code's exponential filter holds the last value across the gaps.

In [ ]:
moving = frames_all[(frames_all.spd > 5) & frames_all.theta.notna()]
print("Heading against the direction of travel (frames with speed > 5 mm/s):")
for name, ang in [("atan2(vy, vx)", np.arctan2(moving.vy, moving.vx)), ("atan2(-vy, vx)", np.arctan2(-moving.vy, moving.vx))]:
    diff = (moving.theta - np.degrees(ang) % 360 + 180) % 360 - 180
    print(f"  theta against {name}: median |difference| {np.median(np.abs(diff)):.1f} deg; {(np.abs(diff) < 30).mean():.3f} of frames within 30 deg")
print("Upwind is 180 deg (code: facing upwind 160 to 200; walking upwind vx < 0). Median vx of frames heading 160-200 deg at speed > 5 mm/s:",
      round(float(moving.vx[(moving.theta > 160) & (moving.theta < 200)].median()), 2), "mm/s")

opts0 = plumes.TurnOptions()
check = []
for s in sessions[:: 6]:
    p = plumes.prepare_frames(s["frames"], opts0)
    tt = plumes.turn_table(p, s["turns"], onset="start")
    check.append(tt.assign(plume=s["info"]["plume"]))
check = pd.concat(check)
print(f"\nTurn events, from every 6th file ({len(check)} turns):")
print("sign of dtheta at the peak of |dtheta| within the event, against turn_direction:")
print(pd.crosstab(np.sign(check.dtheta_peak), check.direction).to_string())
print("event duration (frames):", ((check.stop - check.start) * 60).round().value_counts().sort_index().to_dict())
lead = ((check.start - check.onset20) * 60).round()
print("frames from the SI onset (20% of the peak on the rising edge, rotation in the turn's direction) to the event start:", lead.describe().round(2).to_dict())
print("share of turns whose run of |dtheta| >= 20% of the peak (the first version's rule) contains rotation the other way:",
      round(float(check.unsigned_run_reverses.mean()), 4), "; their onset moves by", 
      ((check.onset20 - check.onset20_unsigned) * 60).round()[check.unsigned_run_reverses].describe().round(1).to_dict(), "frames")
diffs = []
for s in sessions[:: 6]:
    fr = s["frames"]
    for r in s["turns"].head(25).itertuples(index=False):
        g = fr[fr.trj == r.trj]
        i = np.searchsorted(g.t.to_numpy(), r.start_time - 1e-9)
        if i < len(g):
            diffs.append(((g.theta_smooth.iloc[i] - r.turn_theta + 180) % 360 - 180, g.x.iloc[i] - r.turn_x, g.y.iloc[i] - r.turn_y))
diffs = np.abs(np.array(diffs))
print(f"turn_theta against theta_smooth, turn_x and turn_y against x and y at the event start ({len(diffs)} turns, the first 25 of each of these files):",
      "median |difference|", np.nanmedian(diffs, 0).round(4).tolist(), "; largest", np.nanmax(diffs, 0).round(4).tolist())

The heading is the direction of travel measured with y reversed, theta = atan2(-vy, vx) (not
atan2(vy, vx)), and 180 degrees is travel towards -x, upwind, as the code assumes; the files say only
"standardized arena coordinates". The turn direction (+1 or -1) is the sign of the rotational velocity
at the event's peak, so +1 turns are counterclockwise in the (x, -y) frame. Whether that is a left or
a right turn depends on how the camera viewed the arena, which the files do not state, and the
direction column is described only as "Turn direction label"; `methods-intensity-right-bias` needs
that and is blocked. The event start is not the SI's onset as computed from the files' dtheta: it comes
a median of 4 frames (67 ms) after the time the rotation in the turn's direction crosses 20% of its peak
(from every 6th file above). The claims file asks for the SI's onset in that case, so it is the main
reading below, and the code's use of the event start is the other. The rule follows the rotation in the
turn's own direction; the first version of this attempt followed |dtheta|, whose run went on into a turn
the other way in 0.2% of the turns (printed above).

**Clock.** The turn events share the trajectories' clock: every event's peak of |dtheta| has the
sign of its direction. The odor signals were computed by the authors from the projected movie, and
the files give no movie frame for a trajectory frame, so whether they share the movie's clock cannot
be shown here.

In [ ]:
# Fig. 5 turn tables: every plume group, both onsets. Frames are prepared once per group.
t0 = time.time()
groups = {"smooth": ["smooth"], "complex_2a": ["complex_2a"], "complex_1a": ["complex_1a"]}
prepared, turns = {}, {}
for g in groups:
    sub = [s for s in sessions if s["info"]["plume"] == g]
    prepared[g] = plumes.prepare_frames(pd.concat([s["frames"] for s in sub], ignore_index=True), opts0)
    turns[g] = pd.concat([s["turns"] for s in sub], ignore_index=True)
prepared["complex_both"] = pd.concat([prepared["complex_1a"], prepared["complex_2a"]], ignore_index=True)
turns["complex_both"] = pd.concat([turns["complex_1a"], turns["complex_2a"]], ignore_index=True)
del frames_all, moving
for s in sessions:
    s["frames"] = None    # free memory: everything below uses the prepared frames
TT = {(g, on): plumes.turn_table(prepared[g], turns[g], onset=on) for g in prepared for on in ("rule20", "start")}
print(f"turn tables built in {time.time() - t0:.0f} s")
for (g, on), tt in TT.items():
    print(f"{g:13s} onset={on:6s} turns {len(tt):6d}, matched to a frame {tt.matched.sum():6d}")

## fig5e-cue-betas

> For upwind-facing flies in the bulk of the plume, the gradient cue 50 ms before a turn significantly
> predicts turn direction in the smooth plume while motion does not, and the motion cue does in the
> complex plume while the gradient does not.

Readings (claims file): heading window in the complex plume (160 to 200, SI; 150 to 210, code); the
code's 15 mm margin (not applied; applied); z-scoring (none, code; pooled over the plume's frames;
per trajectory); "not significant" (p >= 0.05; p >= 0.01). Onset: the SI's 20% rule, which the claims
file asks for when the event start differs (main), and the event start (code, a variant).

Which complex files are the paper's complex plume is not stated in the paper. Assumed: the smoke-1a
files, as the dandiset's description says the smoke_1a_orig plume "was analyzed throughout the paper"
and the session descriptions of these files name that movie (Data check). The code selects "smoke-2a"
(`metadata/published_panel_params.json`); those files and both sets together are shown. The code's
published tables are compared below with every reading, to show which files and readings they come
from. Expected values are compared, as the criterion says, under the code's readings (event start,
margin, no z-scoring, complex heading 150 to 210, smoke-2a).

In [ ]:
rows = []
fits5 = {}
for (g, on), tt in TT.items():
    headings = [(160.0, 200.0)] if g == "smooth" else [(160.0, 200.0), (150.0, 210.0)]
    for hd in headings:
        for margin in (False, True):
            for z in ("none", "pooled", "per_trajectory"):
                o = plumes.TurnOptions(heading=hd, margin=margin, zscore=z)
                k = plumes.keep(tt, o)
                r = plumes.fig5_models(k)
                f = r["fits"]["full"]
                fits5[(g, on, hd, margin, z)] = r
                rows.append({"group": g, "onset": on, "heading": f"{hd[0]:.0f}-{hd[1]:.0f}", "margin": margin, "z": z,
                             "turns": int(f.nobs), "trajectories": k.cluster.nunique(),
                             "b_grad": f.params["gradient"], "se_grad": f.bse["gradient"], "p_grad": f.pvalues["gradient"],
                             "b_motion": f.params["motion"], "se_motion": f.bse["motion"], "p_motion": f.pvalues["motion"],
                             "b_int": f.params["intensity"], "p_int": f.pvalues["intensity"],
                             "dominance": r["dominance"], **{f"ll_{m}": v for m, v in r["ll"].items()}})
R5 = pd.DataFrame(rows)
print(R5.drop(columns=[c for c in R5 if c.startswith("ll_")]).round(4).to_string(index=False))

In [ ]:
# Which complex files and readings give the code's published tables? Those tables (the claims file's
# expected values) are read from the paper's repository at the pinned commit, for this comparison only.
import io as _io, requests
PUB = "https://raw.githubusercontent.com/emonetlab/GradientMotionMultiplePlumes/effc7b0317f6d3b62136c2a07bcba3e80e778575/data/published_panel_tables/"
pub = {pl: (pd.read_csv(_io.StringIO(requests.get(PUB + f"fig5e_cue_beta_{pl}_plume.csv", timeout=60).text)).iloc[0],
            pd.read_csv(_io.StringIO(requests.get(PUB + f"fig5f_s5_model_comparison_{pl}_plume.csv", timeout=60).text)).set_index("variant").log_likelihood_mean)
       for pl in ("smooth", "complex")}
names = {"mother": "full", "grad_only": "gradient", "vel_only": "motion", "joint_drop": "base"}
for pl, (beta, ll) in pub.items():
    print(f"published {pl}: motion {beta.beta_motion:+.4f} (p {beta.p_motion:.4f}), gradient {beta.beta_gradient:+.4f} (p {beta.p_gradient:.4f}); "
          + ", ".join(f"LL {names[k]} {v:.4f}" for k, v in ll.items()))
cmp_rows = []
for _, r in R5[R5.z == "none"].iterrows():
    beta, ll = pub["smooth" if r.group == "smooth" else "complex"]
    cmp_rows.append({"group": r.group, "onset": r.onset, "heading": r.heading, "margin": r.margin, "turns": r.turns,
                     "|d motion| / SE": abs(r.b_motion - beta.beta_motion) / beta.beta_motion_se,
                     "|d gradient| / SE": abs(r.b_grad - beta.beta_gradient) / beta.beta_gradient_se,
                     "max |d LL|": max(abs(r[f"ll_{names[k]}"] - v) for k, v in ll.items())})
CMP = pd.DataFrame(cmp_rows)
print("\nDistance of each reading (predictors as stored) from the published tables:")
print(CMP[~CMP.margin].round(4).to_string(index=False))

In [ ]:
# The two complex sets by projection flip (SI onset and file start, heading 160-200, as stored)
desc_of = {s["info"]["session_id"]: s["info"]["description"] for s in sessions}
rows = []
for (g, on), tt in TT.items():
    if g == "complex_both":
        continue
    tt = tt.assign(flip=tt.session.map(lambda s: "flipped" if ("vflip1" in desc_of[s] or "vflipTrue" in desc_of[s]) else "not flipped"))
    for fl, sub in tt.groupby("flip"):
        k = plumes.keep(sub, plumes.TurnOptions())
        f = plumes.fit_logit(k, plumes.MODELS["full"])
        rows.append({"group": g, "onset": on, "projection": fl, "turns": int(f.nobs), "files": k.session.nunique(),
                     "b_motion": f.params["motion"], "p_motion": f.pvalues["motion"], "b_grad": f.params["gradient"],
                     "p_grad": f.pvalues["gradient"], "b_int": f.params["intensity"], "p_int": f.pvalues["intensity"]})
print("Full model by projection flip:")
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# The sign of the gradient cue against each set's intensity profile, and the layout of the profiles
bins = np.arange(0, 190, 10)
def flip_of(s):
    return "flipped" if ("vflip1" in desc_of[s] or "vflipTrue" in desc_of[s]) else "not flipped"
geo, prof = [], {}
for g in ["smooth", "complex_1a", "complex_2a"]:
    p = prepared[g]
    for hname, hm in (("upwind 160-200", p.theta.between(160, 200)), ("downwind 340-20", (p.theta >= 340) | (p.theta <= 20))):
        u = p[hm & p.x.between(30, 220) & p.y.between(0, 185)]
        u = u.assign(flip=u.session.map(flip_of), yb=pd.cut(u.y, bins))
        for fl, uf in u.groupby("flip"):
            tab = uf.groupby("yb", observed=False).agg(I=("signal", "mean"), G=("spatial_gradient", "mean"), n=("y", "size"))
            dIdy = np.gradient(tab.I.to_numpy(), 10.0)
            ok = (tab.n.to_numpy() > 500) & np.isfinite(dIdy) & np.isfinite(tab.G.to_numpy())
            rf = []
            for _, us in uf.groupby("session"):
                t2 = us.groupby("yb", observed=False).agg(I=("signal", "mean"), G=("spatial_gradient", "mean"), n=("y", "size"))
                d2 = np.gradient(t2.I.to_numpy(), 10.0)
                o2 = (t2.n.to_numpy() > 100) & np.isfinite(d2) & np.isfinite(t2.G.to_numpy())
                if o2.sum() >= 5:
                    rf.append(np.corrcoef(t2.G.to_numpy()[o2], d2[o2])[0, 1])
            rf = np.array(rf)
            geo.append({"group": g, "heading": hname, "projection": fl, "frames": len(uf), "bins": int(ok.sum()),
                        "r(mean gradient, d mean intensity / dy)": np.corrcoef(tab.G.to_numpy()[ok], dIdy[ok])[0, 1],
                        "slope": np.polyfit(dIdy[ok], tab.G.to_numpy()[ok], 1)[0],
                        "files": len(rf), "median r per file": np.nanmedian(rf), "files with r < 0": int((rf < 0).sum())})
            if hname.startswith("up"):
                prof[(g, fl)] = tab.I
GEO = pd.DataFrame(geo)
print("Mean gradient cue against the slope of the mean intensity across y (10 mm bins with > 500 frames, 30 <= x <= 220 mm):")
print(GEO.round(3).to_string(index=False))
PROF = pd.DataFrame(prof)
PROF.index = [f"{iv.left:.0f}-{iv.right:.0f}" for iv in PROF.index]
print("\nMean intensity by y (mm), upwind-facing frames:")
print(PROF.round(1).to_string())
print("y bin of the highest mean intensity:", PROF.idxmax().to_dict())
pc = {f"smoke-1a {a} with smoke-2a {b}": float(np.corrcoef(PROF[("complex_1a", a)], PROF[("complex_2a", b)])[0, 1])
      for a in ("not flipped", "flipped") for b in ("not flipped", "flipped")}
print("Correlation of the intensity profiles across y:", {k: round(v, 3) for k, v in pc.items()})

In [ ]:
def smooth_pass(r, alpha):
    return (r.p_grad < 0.005) and (r.p_motion >= alpha)
def complex_pass(r, alpha):
    return (r.p_motion < 0.01) and (r.p_grad >= alpha)

summ = []
for g in ["smooth", "complex_2a", "complex_1a", "complex_both"]:
    sub = R5[R5.group == g]
    for alpha in (0.05, 0.01):
        ok = sub.apply(lambda r: smooth_pass(r, alpha) if g == "smooth" else complex_pass(r, alpha), axis=1)
        summ.append({"group": g, "not significant at": f"p >= {alpha}", "readings": len(sub), "pass": int(ok.sum())})
print("Readings in which each plume's part of the criterion holds:")
print(pd.DataFrame(summ).to_string(index=False))

# the whole criterion, both plumes paired by onset, margin and z-scoring
both = []
for g in ["complex_1a", "complex_2a", "complex_both"]:
    for _, c in R5[R5.group == g].iterrows():
        sm = R5[(R5.group == "smooth") & (R5.onset == c.onset) & (R5.margin == c.margin) & (R5.z == c.z)].iloc[0]
        for alpha in (0.05, 0.01):
            both.append({"complex files": g, "onset": c.onset, "heading": c.heading, "margin": c.margin, "z": c.z, "alpha": alpha,
                         "smooth part": smooth_pass(sm, alpha), "complex part": complex_pass(c, alpha)})
BOTH = pd.DataFrame(both)
BOTH["criterion"] = BOTH["smooth part"] & BOTH["complex part"]
print("\nWhole criterion, readings that pass / readings run, by complex files and onset:")
print(BOTH.groupby(["complex files", "onset"]).criterion.agg(["sum", "count"]).to_string())
print("\nPassing readings:")
print(BOTH[BOTH.criterion].drop(columns=["smooth part", "complex part", "criterion"]).drop_duplicates(["complex files", "onset", "heading", "z"]).to_string(index=False))

print("\nComplex plume, motion coefficient over all readings: range", R5[R5.group != "smooth"].groupby("group").b_motion.agg(["min", "max"]).round(4).to_dict("index"))
print("Complex plume, smallest p of the motion coefficient per group:", R5[R5.group != "smooth"].groupby("group").p_motion.min().round(4).to_dict())
print("Smooth plume, largest p of the gradient coefficient:", round(R5[R5.group == "smooth"].p_grad.max(), 5),
      "; smallest p of motion:", round(R5[R5.group == "smooth"].p_motion.min(), 4))

code_rd = lambda g, hd: R5[(R5.group == g) & (R5.onset == "start") & R5.margin & (R5.z == "none") & (R5.heading == hd)].iloc[0]
cs, cc = code_rd("smooth", "160-200"), code_rd("complex_2a", "150-210")
exp = [("gradient, smooth", cs.b_grad, -0.1181, 0.0331), ("motion, smooth", cs.b_motion, -0.0074, 0.0114),
       ("motion, complex", cc.b_motion, 0.0441, 0.0159), ("gradient, complex", cc.b_grad, 0.0185, 0.0161)]
print("\nUnder the code's readings (event start, margin, no z-scoring, complex smoke-2a at 150-210):")
for name, got, want, tol in exp:
    print(f"  {name:18s} measured {got:+.4f}  expected {want:+.4f} +/- {tol:.4f}  within: {abs(got - want) <= tol}")
print(f"  turns kept: smooth {cs.turns}, complex {cc.turns}; p: smooth gradient {cs.p_grad:.5f}, smooth motion {cs.p_motion:.3f}, "
      f"complex motion {cc.p_motion:.3f}, complex gradient {cc.p_grad:.3f}")

**Result: not reproduced.** The smoke-1a files are assumed to be the paper's complex plume, but no
file set or onset gives the result in every reading, so the outcome does not rest on that choice. With
the criterion's onset (the SI's 20% rule)
and the smoke-1a files, no reading passes (0 of 24): the complex motion coefficient has p 0.046 at 160
to 200 degrees and 0.090 at 150 to 210 (as stored or pooled), above 0.01, and with per-trajectory
z-scoring, where it has p 0.0046 and 0.013, the smooth motion coefficient becomes significant (p
0.0079). Under the other z-scorings the smooth part holds (gradient p below 0.0001, motion p 0.23).
With the code's smoke-2a files the criterion holds in 16 of 24 readings (as stored or pooled) and fails
with per-trajectory z-scoring (complex motion p 0.098 and 0.20), so not every reading gives it there
either, and the smoke-2a motion coefficient is negative (-0.028 and -0.024 as stored), opposite to the
published +0.044. The two sets together pass no reading. Within each set the motion coefficient has the
same sign in flipped and unflipped trials (smoke-1a +0.049 and +0.020, smoke-2a -0.028 and -0.030, SI
onset), so the projection flip does not reverse the sign within a set. Whether the cues of one set
carry a mirrored sign is not settled by these outputs. In the smooth plume the mean gradient cue
follows the slope of the mean intensity across y (r -0.96 and -0.94 facing upwind, +0.93 and +0.94
facing downwind; facing upwind 71 of 72 files have a negative r), but in neither complex set does it
(r -0.65 to +0.33, per-file medians -0.19 to +0.19), so this test cannot tell the complex sets apart.
Their projections are not mirrored relative to each other: smoke-2a's mean intensity across y
correlates with smoke-1a's of the same projection flip (r 0.77 and 0.67), not with the opposite one
(0.04 and -0.03). With the dataset's turn start (the code's
onset) and the smoke-1a files the criterion holds at 160 to 200 as stored or pooled (complex motion p
0.0059, gradient p 0.28; smooth gradient p 0.0008, motion p 0.35) and fails at 150 to 210 and per
trajectory. That reading is the one nearest the code's published tables: its complex coefficients are
within 0.1 SE of them and its four log-likelihoods within 0.001 per turn. The smoke-1a files with the
SI onset at 160 to 200 are also close (0.45 SE in motion, 0.09 SE in gradient, log-likelihoods within
0.0036), while the code's own reading (smoke-2a, 150 to 210) is 3.5 SE away in motion. Under the code's readings the smooth coefficients are
within tolerance and the complex ones are not (motion -0.0119 against +0.0441 +/- 0.0159, gradient
-0.0124 against +0.0185 +/- 0.0161). Whether the dataset's turn start is the SI's onset is asked of a
reader.

### methods-intensity-right-bias (blocked claim)

The intensity coefficient of the full model (heading 160 to 200, no margin), in the direction of each
turn code. The files do not say which code is a right turn (Data check), so the criterion is evaluated
under both mappings.

In [ ]:
rows = []
for g in ["smooth", "complex_1a", "complex_2a"]:
    for on in ("rule20", "start"):
        for z in ("none", "per_trajectory"):
            r = R5[(R5.group == g) & (R5.onset == on) & (R5.heading == "160-200") & ~R5.margin & (R5.z == z)].iloc[0]
            rows.append({"group": g, "onset": on, "z": z, "turns": r.turns, "b_intensity": r.b_int, "p": r.p_int,
                         "favours": ("+1" if r.b_int > 0 else "-1") if r.p_int < 0.05 else "neither (p >= 0.05)"})
IB = pd.DataFrame(rows)
print(IB.round(5).to_string(index=False))
for right in ("-1", "+1"):
    def holds(g, on, z):
        a = IB[(IB.group == "smooth") & (IB.onset == on) & (IB.z == z)].iloc[0]
        b = IB[(IB.group == g) & (IB.onset == on) & (IB.z == z)].iloc[0]
        left = "+1" if right == "-1" else "-1"
        return ((a.favours == right and b.favours != left) or (b.favours == right and a.favours != left))
    res = {f"{g}, {on}, {z}": holds(g, on, z) for g in ("complex_1a", "complex_2a") for on in ("rule20", "start") for z in ("none", "per_trajectory")}
    print(f"If {right} is a right turn, criterion holds:", res)

## fig5f-cue-dominance

> Cue dominance, the gradient-minus-motion log-likelihood difference relative to the total bilateral
> contribution, is near 1 in the smooth plume (gradient dominates) and negative in the complex plume
> (motion dominates).

The four models and the measure are the code's (`VARIANT_FORMULAS`); the readings are those of
fig5e-cue-betas, paired so that both plumes use the same onset, margin and z-scoring.

In [ ]:
ll_cols = ["ll_full", "ll_gradient", "ll_motion", "ll_base"]
print("Mean log-likelihood per turn and dominance, every reading:")
print(R5[["group", "onset", "heading", "margin", "z", "turns"] + ll_cols + ["dominance"]].round(4).to_string(index=False))

pairs = []
for g in ["complex_2a", "complex_1a", "complex_both"]:
    for _, c in R5[R5.group == g].iterrows():
        s = R5[(R5.group == "smooth") & (R5.onset == c.onset) & (R5.margin == c.margin) & (R5.z == c.z)].iloc[0]
        pairs.append({"complex files": g, "onset": c.onset, "complex heading": c.heading, "margin": c.margin, "z": c.z,
                      "smooth": s.dominance, "complex": c.dominance,
                      "signs hold": (s.dominance > 0) and (c.dominance < 0),
                      "both within tolerance": abs(s.dominance - 0.946) <= 0.095 and abs(c.dominance + 0.662) <= 0.066})
PR = pd.DataFrame(pairs)
print("\nPaired readings (expected: smooth 0.946 +/- 0.095, complex -0.662 +/- 0.066):")
print(PR.round(3).to_string(index=False))
print("\nReadings where the signs hold / both values within tolerance, by complex file set:")
print(PR.groupby("complex files")[["signs hold", "both within tolerance"]].agg(["sum", "count"]).to_string())
print("\nUnder the code's readings: smooth", round(cs.dominance, 3), ", complex (smoke-2a)", round(cc.dominance, 3))
print("Smooth dominance over all readings: range", round(R5[R5.group == 'smooth'].dominance.min(), 3), "to", round(R5[R5.group == 'smooth'].dominance.max(), 3))

**Result: partially reproduced, resting on an assumption.** With the criterion's onset and the smoke-1a
files (the readings of fig5e-cue-betas), the signs hold in 4 of the 6 readings (heading 160 to 200
under every z-scoring, and 150 to 210 per trajectory; the margin changes nothing), and the two values
are never within tolerance together: smooth 0.937 and complex -0.492 as stored or pooled at 160 to 200
(complex outside -0.662 +/- 0.066), smooth 0.518 and complex -0.716 per trajectory (smooth outside
0.946 +/- 0.095); at 150 to 210 as stored the complex value is +0.114. Both values are within tolerance
with the smoke-2a files at 150 to 210 as stored or pooled (0.937 and -0.717), and with the dataset's
turn start and the smoke-1a files at 160 to 200 (0.879 and -0.701), the reading nearest the code's
published tables. Under the code's readings the complex value is +0.388. The complex log-likelihoods
differ little (full minus base 0.011 per turn with smoke-1a at 160 to 200). So the outcome rests on the
onset as well as on the file set: with the smoke-1a files it is reproduced under the dataset's turn
start, and a reader has been asked whether that start is the SI's onset.

## figs5c-turn-angles

> The detected saccade-like turns reorient flies by about 30 degrees, with median angular
> displacements of -34.2 and +33.8 degrees for the two turn directions.

Split by the sign of dtheta at the peak (Fig. S5C legend). Readings (claims file): all detected turns
of both plumes pooled, or the turns kept in fig5e; displacement as the heading change from the event's
start to its end, or the integral of rotational velocity over the event. The files' events start a
median of 4 frames after the SI's turn onset (Data check), so the event as the paper bounds it is not
the files' event. The integral is therefore also run over the turn as the SI bounds its onset, from the
20% crossing on the rising edge to the 20% crossing on the falling edge of the rotation in the turn's
direction (the falling edge is not stated; taken as the mirror of the onset rule, an assumption). The
integral over the run of that rotation at or above half its peak is shown too, since Fig. S5A measures
a turn's peak width at half height. The first version's run of |dtheta| is shown beside it. The complex files are smoke-1a (as fig5e), with
smoke-2a and both shown.

In [ ]:
def medians(tt):
    disp = (tt.theta_stop - tt.theta_start + 180) % 360 - 180
    out = {}
    for name, v in [("heading change", disp), ("integral of dtheta", tt.int_dtheta), ("integral over the SI-bounded turn", tt.int_dtheta_20), ("integral over the half-height run", tt.int_dtheta_50),
                    ("SI-bounded, first version's |dtheta| run", tt.int_dtheta_20_unsigned)]:
        out[name] = (round(float(v[tt.dtheta_peak < 0].median()), 1), round(float(v[tt.dtheta_peak > 0].median()), 1))
    return out

main_s = plumes.TurnOptions()
sets = {
    "all turns, smooth + complex smoke-1a": pd.concat([TT[("smooth", "start")], TT[("complex_1a", "start")]]),
    "all turns, smooth + complex smoke-2a": pd.concat([TT[("smooth", "start")], TT[("complex_2a", "start")]]),
    "all turns, smooth + both complex": pd.concat([TT[("smooth", "start")], TT[("complex_both", "start")]]),
    "kept turns (SI onset, 160-200, no margin), smooth + smoke-1a": pd.concat([plumes.keep(TT[("smooth", "rule20")], main_s), plumes.keep(TT[("complex_1a", "rule20")], main_s)]),
    "kept turns (SI onset, 160-200, no margin), smooth + smoke-2a": pd.concat([plumes.keep(TT[("smooth", "rule20")], main_s), plumes.keep(TT[("complex_2a", "rule20")], main_s)]),
}
rows = []
for name, tt in sets.items():
    for meas, (neg, pos) in medians(tt).items():
        rows.append({"turns": name, "n": len(tt), "measure": meas, "median, negative": neg, "median, positive": pos,
                     "within tolerance": abs(neg + 34.2) <= 3.42 and abs(pos - 33.8) <= 3.38})
print(pd.DataFrame(rows).to_string(index=False))
for g in ["smooth", "complex_1a", "complex_2a"]:
    print(g, "by group, all turns:", medians(TT[(g, "start")]))
allt = pd.concat([TT[("smooth", "start")], TT[("complex_both", "start")]])
d = (allt.theta_stop - allt.theta_start + 180) % 360 - 180
print("Range of the heading change over all turns:", round(d.min(), 1), "to", round(d.max(), 1), "deg; 5th and 95th percentiles of |change|:",
      np.percentile(np.abs(d), [5, 95]).round(1))
print("Range of the SI-bounded integral:", round(allt.int_dtheta_20.min(), 1), "to", round(allt.int_dtheta_20.max(), 1), "deg")
allg = pd.concat([TT[(g, "start")] for g in ("smooth", "complex_1a", "complex_2a")])
q3 = lambda v: (v * 60).round().quantile([0.25, 0.5, 0.75]).tolist()
print(f"\nEvent bounds against the crossings of the rotation in the turn's direction (frames; quartiles over all {len(allg)} turns of every group):")
print("  event start minus the 20% crossing on the rising edge:", q3(allg.start - allg.onset20))
print("  event start minus the 50% crossing on the rising edge:", q3(allg.start - allg.start50))
print("  event stop minus the 50% crossing on the falling edge:", q3(allg.stop - allg.end50))
print("  event stop minus the 20% crossing on the falling edge:", q3(allg.stop - allg.end20))
print("  share of events that stop before the 20% crossing on the falling edge:", round(float((allg.stop < allg.end20 - 1e-9).mean()), 3))
print("  length, stop minus start: event", q3(allg.stop - allg.start), "; half-height run", q3(allg.end50 - allg.start50),
      "; 20% run", q3(allg.end20 - allg.onset20))


**Result: reproduced, resting on an assumption; a change to not reproduced is proposed.** Over the
files' events, the two displacements the claims file names, the medians are -19.6 to -21.4 and +20.7
to +22.0 degrees for every turn set, outside -34.2 +/- 3.42 and +33.8 +/- 3.38. Integrated over the
run at or above half the peak (Fig. S5A) they are -25.0 to -26.6 and +26.8 to +27.7, also outside.
Only the turn bounded by the 20% crossings on both edges gives medians within tolerance: -34.3 and
+35.2 degrees for all turns of the smooth and smoke-1a files (-35.0 and +35.9 with smoke-2a, -35.0 and
+35.8 with both), and -32.7 and +36.2 for the kept turns (-33.1 and +36.7 with smoke-2a). The medians
are the same with the first version's run of |dtheta|. The files' events lie inside the half-height
run: they start a median of 1 frame after its rising crossing and stop 1 frame before its falling one
(median length 7 frames, against 10 for the half-height run and 16 for the 20% run), and 99.9% stop
before the 20% crossing on the falling edge (a median of 4 frames before). The 20% end is not stated
in the paper and was adopted after the event bounds failed. Under the claims file's readings, which
integrate over the event, the outcome is not reproduced; that change is proposed for the next round,
and whether the dataset's events are the paper's turn bounds is asked of a reader.

## The complex plume movie

One streaming pass gives the mean intensity map and the time-averaged cue maps of Fig. S1F-G; a
second samples the antennae at the Fig. 2 locations, which are drawn from the mean map. See
`plumes.movie_pass1` and `plumes.sample_antennas` for the definitions.

In [ ]:
t0 = time.time()
movie, movie_file = plumes.open_movie(assets, *plumes.COMPLEX_MOVIE)
print("movie", movie.shape, movie.dtype, "chunks", movie.chunks)
P1 = plumes.movie_pass1(movie)
mean_map = P1["mean"]
print(f"pass 1: {P1['n_frames']} frames in {time.time() - t0:.0f} s")
print("intensity: mean map range", round(mean_map.min(), 2), "to", round(mean_map.max(), 2),
      "; values over all frames:", P1["frame_min"], "to", P1["frame_max"])
print("SI: 1500 frames of 1088 x 1728 pixels, 0 to 255; centerline row", plumes.COMPLEX_CENTERLINE)
print("provenance (scratch/figure_s1_window_provenance/source_preprocessed_h5):",
      [plumes._text(v) for v in np.atleast_1d(movie_file["scratch/figure_s1_window_provenance/source_preprocessed_h5"][()])])
print("row of the maximum of the mean map over the first 100 columns (source end):", int(np.argmax(mean_map[:, :100].mean(1))),
      "; over the first 10 columns:", int(np.argmax(mean_map[:, :10].mean(1))))
print("1st and 99th percentiles of the intensity (every 25th frame):", plumes.hist_percentiles(*P1["hist"]))
from skimage.filters import threshold_otsu
print("Otsu threshold of the mean map:", round(float(threshold_otsu(mean_map)), 2))
masks = {"mean > 5": mean_map > 5,
         "Otsu, bias 48 subtracted": plumes.otsu_mask(mean_map, 48, "subtract"),
         "Otsu, bias 48 %": plumes.otsu_mask(mean_map, 48, "percent")}
for k, m in masks.items():
    print(f"  valid region '{k}': {m.mean():.3f} of pixels")

The stored window matches the SI and the dandiset's description (1500 frames, 1088 x 1728, 0 to 255).
Its provenance (printed above) names a preprocessing run "bgsub_gauss3_fullslice_p001_p99", so it may differ from the
movie the paper analysed (the SI describes a linear rescaling from 0-120 to 0-255 only); this cannot be
undone here. The row of maximum mean intensity at the source end is 571 (first 100 columns) or 563
(first 10), not the SI's centerline row 590; the Fig. 2 labels are also run with row 571, as a variant.

## figs1fg-resolution-robustness

> The spatial organization of the gradient and motion cues in the complex plume stays consistent when
> sensor spacing is increased from 3 to 12 pixels or the motion delay is lengthened to about 150 ms,
> changing only when sampled more coarsely than the smooth plume.

Gradient of spacing d: the time average of B - A for sensors d rows apart. The motion cue for a
spacing and a delay is not defined in the paper; three forms are run (`plumes.movie_pass1`): the Fig. 1
cue -(dI/dy)(dI/dt) with centered differences, the spacing and the delay widened (main, as Fig. S1G is
the motion cue of column iv, `<-dI/dy dI/dt>_t`); the same with a backward difference in time (the
first version's form); and the two-sensor correlator A(t - tau)B(t) - A(t)B(t - tau). Each is computed
from the frames as stored and, as a variant, from frames smoothed by the antennae's Gaussian (SD 1.5
pixels), as the code smooths its Fig. 1 cue panels; the paper does not say whether Fig. S1F-G is
smoothed. Pearson r against the 3-pixel map (gradient) and the 3-pixel, 1-frame map (motion) of the
same reading, over every pixel of every 4th column, and over the Otsu mask.

In [ ]:
def r_(a, b, m=None):
    a, b = a.ravel(), b.ravel()
    if m is not None:
        a, b = a[m.ravel()], b[m.ravel()]
    return float(np.corrcoef(a, b)[0, 1])

mask_sub = masks["Otsu, bias 48 subtracted"][:, ::P1["col_step"]][6:-7]
grad_rows = []
for d in plumes.SPACINGS:
    grad_rows.append({"spacing px": d, "r": r_(P1["grad"][(3, False)], P1["grad"][(d, False)]),
                      "r in mask": r_(P1["grad"][(3, False)], P1["grad"][(d, False)], mask_sub),
                      "r, frames smoothed sigma 1.5 (variant)": r_(P1["grad"][(3, True)], P1["grad"][(d, True)])})
G = pd.DataFrame(grad_rows)
print("Gradient maps against the 3-pixel map:")
print(G.round(3).to_string(index=False))
mot, checks = {}, []
for form in plumes.MOTION_FORMS:
    for sm in (False, True):
        tab = pd.DataFrame({tau: {d: r_(P1["motion"][(form, 3, 1, sm)], P1["motion"][(form, d, tau, sm)]) for d in plumes.SPACINGS} for tau in plumes.DELAYS})
        tabm = pd.DataFrame({tau: {d: r_(P1["motion"][(form, 3, 1, sm)], P1["motion"][(form, d, tau, sm)], mask_sub) for d in plumes.SPACINGS} for tau in plumes.DELAYS})
        mot[(form, sm)] = tab
        label = f"{form}, " + ("frames smoothed sigma 1.5" if sm else "frames as stored")
        print(f"\nMotion maps ({label}) against the 3-pixel, 1-frame map; rows spacing (px), columns delay (frames):")
        print(tab.round(3).to_string())
        print("  in the Otsu mask:")
        print(tabm.round(3).to_string())
        gcol = "r, frames smoothed sigma 1.5 (variant)" if sm else "r"
        checks.append({"motion form": form, "frames": "smoothed" if sm else "as stored",
                       "gradient r >= 0.8 at every spacing": bool((G[gcol] >= 0.8).all()),
                       "min gradient r": round(float(G[gcol].min()), 3),
                       "motion r >= 0.8 for delays 1, 2, 4": bool((tab[[1, 2, 4]] >= 0.8).all().all()),
                       "min motion r, delays 1-4": round(float(tab[[1, 2, 4]].min().min()), 3),
                       "8 and 15 below 4 at every spacing": bool(((tab[8] < tab[4]) & (tab[15] < tab[4])).all())})
CK = pd.DataFrame(checks)
CK["criterion"] = CK.iloc[:, 2] & CK.iloc[:, 4] & CK.iloc[:, 6]
print("\nCriterion by reading:")
print(CK.to_string(index=False))
print("Figure values (claims file): gradient r about 0.89 at 12 px; motion r near 1 at 1, 2, 4 frames, about 0.65 at 8 and 0.35 at 15.")

**Result: not reproduced.** With the frames as stored the gradient map's correlation with the 3-pixel
map is 0.817, 0.703 and 0.613 at 6, 9 and 12 pixels, below 0.8 at 9 and 12 pixels, in the whole frame
and in the mask. With every frame smoothed (variant) it is 0.968, 0.911 and 0.809, so the gradient part
holds only with smoothing. The motion part fails under every reading: at a 4-frame delay r is 0.729 to
0.820 with the frames as stored and 0.780 to 0.855 smoothed, below 0.8 at 3 pixels in each form (the
lowest r over delays 1 to 4 is 0.729 to 0.784 across the six readings), where Fig. S1G shows r near 1.
The drop at 8 and 15 frames below the 4-frame value holds in every reading. The centered and correlator
forms differ by at most 0.001 in every value printed.

## Parts of figs1-antisymmetry-index and figs1-temporal-statistics (blocked claims)

Both claims need the smooth plume, which could not be read; their other parts are computed here. For
the three freestream plumes the dandiset holds, in a compact asset, the mean vertical gradient and
motion-cue maps of the Fig. S1 panels and the frequency and intermittency points Fig. S1 plots (a
seeded sample of 1000 pixels per plume). Their Otsu masks and centerlines need the mean intensity, so
every 25th frame (60 of 1500) of each 8 GB freestream window is read, with the frame after it and the
one before, which also gives a check of the scale of the deposited maps. For the complex plume the
maps, the per-pixel maximum and the encounter statistics come from the two passes over its window
(every 4th column). The centerline of a freestream plume is the row of maximum mean intensity over the
10 columns at the source (claims file); for the complex plume, row 590 (SI), and the row found in the
same way as a variant.

In [ ]:
import h5py
fs_path = "sub-figs1-freestream-plumes/sub-figs1-freestream-plumes_ses-figure-s1_behavior.nwb"
fsf = h5py.File(_io.BytesIO(plumes.fetch(assets.set_index("path").asset_id[fs_path])), "r")
arr = fsf["processing/figure_s1_freestream_plumes/figure_s1_image_arrays"]
sc = fsf["scratch/figure_s1_exact_plotted_scatter_points"]
SC = pd.DataFrame({k: [v.decode() if isinstance(v, bytes) else v for v in sc[k][:]] for k in ("plume_id", "frequency_hz", "intermittency_fraction")})
SPEEDS = {"a0001": 5, "a0002": 10, "a0003": 20}
t0 = time.time()
FS = {}
for code_, speed in SPEEDS.items():
    p = next(p for p in assets.path if f"freestream-{code_}" in p and p.endswith("_image.nwb"))
    ds, fh = plumes.open_movie(assets, p, f"scratch/{code_}_figs1_standardized_window_frames")
    if code_ == "a0001":
        print("provenance of the 5 cm/s window (source_remote_npy):", [plumes._text(v) for v in np.atleast_1d(fh["scratch/figure_s1_window_provenance/source_remote_npy"][()])])
    smp = plumes.freestream_sample(ds)
    FS[code_] = {"G": arr[f"{code_}_gradient_y_mean"][:].astype(float), "M": arr[f"{code_}_motion_cue_y_mean"][:].astype(float),
                 **smp, "shape": ds.shape}
    fh.close()
print(f"freestream samples read in {time.time() - t0:.0f} s")
print("provenance of the deposited maps (scratch/figure_s1_provenance/figure_sidecar):", sorted({plumes._text(v) for v in fsf["scratch/figure_s1_provenance/figure_sidecar"][()]}))
for code_, d in FS.items():
    fin = np.isfinite(d["G"]) & np.isfinite(d["M"])
    sl_g = np.polyfit(d["G"][fin], d["grad1"][fin], 1)[0]
    sl_m = np.polyfit(d["M"][fin], d["motion1"][fin], 1)[0]
    print(f"{SPEEDS[code_]:2d} cm/s: window {d['shape']}, {d['n_triplets']} triplets; deposited maps finite on {fin.mean():.3f} of pixels; "
          f"slope of own one-pixel estimate on the deposited map: gradient {sl_g:.3f} (r {np.corrcoef(d['G'][fin], d['grad1'][fin])[0, 1]:.3f}), "
          f"motion {sl_m:.3f} (r {np.corrcoef(d['M'][fin], d['motion1'][fin])[0, 1]:.3f}); mean intensity {d['mean'].min():.1f} to {d['mean'].max():.1f}, "
          f"1st/99th percentiles {plumes.hist_percentiles(*d['hist'])}")

The deposited maps scale with the one-pixel cues computed from the windows by similar factors, 0.169
to 0.172 for the gradient (r 0.94 to 0.95) and 0.196 to 0.221 for motion, whose fits are loose (r 0.24
to 0.46), so the two carry roughly the same units of space and time and their ratio, which the index
takes, is roughly on the scale of the cues computed from the complex window. The provenance printed
above names a "bgsub_gauss3" figure run for the deposited maps and a "bgsub_gauss3_fullslice_p001_p99"
run for the 5 cm/s window, the run the complex window names; that the frames were smoothed by a
Gaussian is read from these names only.

In [ ]:
from skimage.filters import threshold_otsu
def fs_index(G, M, mean, center, bias, mode):
    mask = plumes.otsu_mask(mean, bias, mode) & np.isfinite(G) & np.isfinite(M)
    sg, sm = plumes.antisymmetry(np.nan_to_num(G), center, mask), plumes.antisymmetry(np.nan_to_num(M), center, mask)
    return sg, sm, mask.mean()

AI = []
for code_, d in FS.items():
    c = int(np.argmax(d["mean"][:, :10].mean(1)))
    lo, hi = plumes.hist_percentiles(*d["hist"])
    for mode in ("subtract", "percent"):
        sg, sm, cover = fs_index(d["G"], d["M"], d["mean"], c, 55, mode)
        AI.append({"plume": f"freestream {SPEEDS[code_]} cm/s", "centerline": c, "bias": mode, "mask share": cover,
                   "s_gradient": sg, "s_motion": sm, "index": np.log2(sm / sg), "index, rescaled 1-99%": np.log2(sm / sg) - np.log2(hi - lo)})
# complex: one-pixel maps of pass 1 (rows 1 to H - 2, every 4th column), centerline 590 (SI) or the row found at the source
cs_ = P1["col_step"]
lo, hi = plumes.hist_percentiles(*P1["hist"])
for c_name, c in (("590 (SI)", plumes.COMPLEX_CENTERLINE), ("row at the source", int(np.argmax(mean_map[:, :10].mean(1))))):
    for mode in ("subtract", "percent"):
        mask = plumes.otsu_mask(mean_map, 48, mode)[1:-1, ::cs_]
        sg = plumes.antisymmetry(P1["grad1"], c - 1, mask)
        sm = plumes.antisymmetry(P1["motion1"], c - 1, mask)
        AI.append({"plume": "complex", "centerline": c, "bias": mode, "mask share": mask.mean(),
                   "s_gradient": sg, "s_motion": sm, "index": np.log2(sm / sg), "index, rescaled 1-99%": np.log2(sm / sg) - np.log2(hi - lo)})
AI = pd.DataFrame(AI)
print("Antisymmetry scores and index log2(s_motion / s_gradient):")
print(AI.round(4).to_string(index=False))
for col in ("index", "index, rescaled 1-99%"):
    for mode in ("subtract", "percent"):
        a = AI[AI.bias == mode].set_index("plume")[col]
        fsv = a[[f"freestream {v} cm/s" for v in (5, 10, 20)]].to_numpy()
        cx = AI[(AI.bias == mode) & (AI.plume == "complex")][col].to_numpy()
        print(f"{col}, bias {mode}: rises 5 < 10 < 20: {bool(fsv[0] < fsv[1] < fsv[2])}; complex above every freestream plume "
              f"(both centerlines): {bool((cx > fsv.max()).all())}")

In [ ]:
# encounter statistics: freestream from the deposited plotted points, complex from pass 2 (computed below; shown here once it has run)
FSQ = SC.assign(speed=SC.plume_id.str.extract(r"_(\d+)cm_s")[0].astype(int)).groupby("speed")[["frequency_hz", "intermittency_fraction"]].agg(["median", "count"])
print("Freestream, deposited Fig. S1 points (median, count):")
print(FSQ.round(3).to_string())

## fig2c-complex-auc and fig2e-complex-weights

> In the complex plume, a logistic model on the odor motion feature predicts the side of the
> centerline better (ROC AUC about 0.62) than one on the gradient feature (about 0.54), and the odor
> sum has no predictive power.

> Fitted on all three features together in the complex plume, the logistic model gives motion a larger
> weight (about 0.40) than the gradient (about 0.10), and the sum a weight of 0; the combined model
> reaches a test AUC of 0.612.

Samples per the SI (Plume movies, Logistic regression models) and the code (centerline_models.py),
as the claims file gives them. Readings named in the claims file: the region (a) mean intensity above
5, (b) the Otsu mask (bias subtracted); the sampling (a) random windows split at random, (b) split by
location. Two mirror-balanced samplings are added (author, step 1.6): each sample paired with its
left-right mirror, label flipped, with the windows split at random ("mirror") or by location
("mirror_location", so that no test window is also a training window). The paper does not describe how
samples are balanced, but it reports a sum AUC of exactly 0.5 with SD 0 and a sum weight of 0 "as
expected", which only a mirror-balanced sample gives (the deposited plume is not symmetric about row
590). A variant labels the side with the centerline at row 571 (see the movie's data check). 10
repeats per reading (the paper uses 100; 1000 new locations and 2,400,000 + 600,000 samples each, as
the SI), seeds fixed.

In [ ]:
rng2 = np.random.default_rng(20260712)
N_REP = 10
regions = {"mean > 5": masks["mean > 5"], "Otsu": masks["Otsu, bias 48 subtracted"]}
loc = {k: [plumes.choose_locations(v, 1000, rng2) for _ in range(N_REP)] for k, v in regions.items()}
rows_all = np.concatenate([r for k in loc for r, c in loc[k]])
cols_all = np.concatenate([c for k in loc for r, c in loc[k]])
t0 = time.time()
traces, ENC = plumes.sample_antennas(movie, rows_all, cols_all, max_map=P1["max"], col_step=P1["col_step"])
print(f"pass 2: antennae at {len(rows_all)} locations over {traces.shape[0]} frames in {time.time() - t0:.0f} s;",
      "signal range", float(traces.min()), "to", float(traces.max()))

In [ ]:
# figs1-temporal-statistics, complex part (blocked claim): per-pixel encounter statistics within the Otsu mask
rows = []
for mode in ("subtract", "percent"):
    m = plumes.otsu_mask(mean_map, 48, mode)[:, ::P1["col_step"]]
    rows.append({"plume": "complex", "mask": f"Otsu, bias 48 {mode}", "pixels": int(m.sum()),
                 "median frequency (Hz)": float(np.median(ENC["frequency"][m])), "median intermittency": float(np.median(ENC["intermittency"][m]))})
for sp, r in FSQ.iterrows():
    rows.append({"plume": f"freestream {sp} cm/s", "mask": "deposited plotted points", "pixels": int(r[("frequency_hz", "count")]),
                 "median frequency (Hz)": r[("frequency_hz", "median")], "median intermittency": r[("intermittency_fraction", "median")]})
TS = pd.DataFrame(rows)
print(TS.round(3).to_string(index=False))
fsq = TS[TS.plume.str.startswith("freestream")].set_index("plume")
print("complex median frequency above every freestream median:", bool((TS[TS.plume == "complex"]["median frequency (Hz)"] > fsq["median frequency (Hz)"].max()).all()),
      "; 20 cm/s frequency above 5 cm/s:", bool(fsq.loc["freestream 20 cm/s", "median frequency (Hz)"] > fsq.loc["freestream 5 cm/s", "median frequency (Hz)"]),
      "; 20 cm/s intermittency below 5 cm/s:", bool(fsq.loc["freestream 20 cm/s", "median intermittency"] < fsq.loc["freestream 5 cm/s", "median intermittency"]))

**Parts of the blocked Fig. S1 claims.** Among the plumes the dandiset holds, the cue antisymmetry index
rises with wind speed (-2.93, -1.53 and +0.04 at 5, 10 and 20 cm/s, bias subtracted) and the complex
plume's is the highest (4.40 with the centerline at row 590, 4.26 at row 563), under both bias readings
and with the intensities rescaled between their 1st and 99th percentiles. The gradient score s_gradient
is 11.7, 16.2 and 16.3 and the motion score s_motion 1.5, 5.6 and 16.8 at 5, 10 and 20 cm/s. The
complex plume's median encounter frequency in its Otsu mask (2.08 Hz with the bias subtracted, 2.28 Hz
as a percentage) is above the freestream medians of the deposited points (0.08, 0.40 and 1.08 Hz), and
the 20 cm/s plume has a higher median frequency and a lower median intermittency (0.433) than the 5
cm/s plume (0.784). The smooth plume's place, which both criteria need, cannot be computed, so both
claims stay blocked.

In [ ]:
t0 = time.time()
fig2, delays, fig571 = [], [], []
for ri, region in enumerate(regions):
    for rep in range(N_REP):
        j = ri * N_REP + rep
        rr, cc = loc[region][rep]
        F = plumes.window_features(traces[:, j * 1000:(j + 1) * 1000], delays=range(1, 16))
        lab = (rr + 1.5 > plumes.COMPLEX_CENTERLINE).astype(int)
        for sampling in ("random", "location", "mirror", "mirror_location"):
            Xtr, ytr, Xte, yte = plumes.draw_samples(F, lab, sampling, rng2)
            fig2.append({"region": region, "sampling": sampling, "centerline": plumes.COMPLEX_CENTERLINE, "repeat": rep, "right share": lab.mean(),
                         **plumes.fit_fig2(Xtr, ytr, Xte, yte)})
        if region == "mean > 5":    # variant: centerline at row 571, the row of maximum mean intensity at the source
            lab571 = (rr + 1.5 > 571).astype(int)
            Xtr, ytr, Xte, yte = plumes.draw_samples(F, lab571, "mirror_location", rng2)
            fig571.append({"region": region, "sampling": "mirror_location", "centerline": 571, "repeat": rep, "right share": lab571.mean(),
                           **plumes.fit_fig2(Xtr, ytr, Xte, yte)})
        if region == "mean > 5":    # the complex part of figs2c-one-frame-delay, shown below
            keys = tuple(f"motion{d}" for d in range(1, 16))
            Xtr, ytr, Xte, yte = plumes.draw_samples(F, lab, "mirror_location", rng2, keys=keys)
            for k, d in enumerate(range(1, 16)):
                delays.append({"repeat": rep, "delay": d, **plumes.fit_fig2(Xtr[:, [k]], ytr, Xte[:, [k]], yte, names=("motion",))})
        del F
F2 = pd.DataFrame(fig2)
print(f"{len(F2)} repeats fitted in {time.time() - t0:.0f} s; no fit failed (sklearn lbfgs, unpenalised)")
agg = F2.groupby(["region", "sampling"]).agg(["mean", "std"]).drop(columns=["repeat"])
agg.columns = [f"{a} {b}" for a, b in agg.columns]
print(agg.round(3).T.to_string())
F571 = pd.DataFrame(fig571)
print("\nVariant, centerline at row 571 (region mean > 5, mirror_location sampling), mean and SD over repeats:")
print(F571.drop(columns=["region", "sampling", "centerline", "repeat"]).agg(["mean", "std"]).round(3).to_string())

In [ ]:
m = F2.groupby(["region", "sampling"]).mean(numeric_only=True)
rows = []
for (region, sampling), r in m.iterrows():
    rows.append({"region": region, "sampling": sampling,
                 "motion AUC": r.auc_motion, "gradient AUC": r.auc_gradient, "sum AUC": r.auc_sum,
                 "motion > gradient": r.auc_motion > r.auc_gradient,
                 "sum within 0.5 +/- 0.05": abs(r.auc_sum - 0.5) <= 0.05,
                 "motion within 0.618 +/- 0.005": abs(r.auc_motion - 0.618) <= 0.005,
                 "gradient within 0.537 +/- 0.012": abs(r.auc_gradient - 0.537) <= 0.012})
print("fig2c-complex-auc, means over repeats:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
rows = []
for (region, sampling), r in m.iterrows():
    rows.append({"region": region, "sampling": sampling, "|w motion|": abs(r.w_motion), "|w gradient|": abs(r.w_gradient), "|w sum|": abs(r.w_sum),
                 "combined AUC": r.auc_all, "|motion| > |gradient|": abs(r.w_motion) > abs(r.w_gradient), "|sum| < 0.01": abs(r.w_sum) < 0.01,
                 "motion within 0.403 +/- 0.034": abs(abs(r.w_motion) - 0.403) <= 0.034,
                 "gradient within 0.098 +/- 0.041": abs(abs(r.w_gradient) - 0.098) <= 0.041,
                 "AUC within 0.612 +/- 0.061": abs(r.auc_all - 0.612) <= 0.061})
print("\nfig2e-complex-weights, means over repeats (weights of standardised features):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSD over repeats of the motion AUC:", F2.groupby(["region", "sampling"]).auc_motion.std().round(4).to_dict())

**Result (fig2c-complex-auc): partially reproduced, resting on an assumption.** Under every reading the
motion model predicts the side better than the gradient model (AUC 0.600 to 0.603 against 0.553 to
0.561). Under the two mirror-balanced samplings the sum has no predictive power (AUC 0.500, SD 0), as in
the paper, and splitting by location so that no test window is a training window changes neither AUC
(motion 0.603 and 0.601, gradient 0.557 and 0.561). Under the two samplings the claims file names the
sum predicts the side (AUC 0.620 to 0.663), because the deposited plume is not symmetric about row 590,
so the criterion's sum condition fails and the outcome would be not reproduced. The motion AUC is below
0.618 +/- 0.005 and the gradient AUC above 0.537 +/- 0.012 under every reading, and with the centerline
at row 571 (0.598 and 0.564), so the values do not hold. The outcome rests on the mirror-balanced
sampling.

**Result (fig2e-complex-weights): partially reproduced.** The |motion| weight (0.414 to 0.430) exceeds
the |gradient| weight (0.231 to 0.281) under every reading, and the sum weight is 0 under the
mirror-balanced readings (0.22 to 0.34 otherwise). The motion weight is within 0.403 +/- 0.034 under
every reading and the combined AUC within 0.612 +/- 0.061 under the mirror-balanced readings (0.610 to
0.614), but the gradient weight is more than twice the expected 0.098 +/- 0.041 under every reading, and
0.276 with the centerline at row 571, so only the ordering holds; every reading gives this outcome.

### Complex part of figs2c-one-frame-delay (blocked claim)

The claim needs both plumes; the smooth plume could not be read, so it is blocked. The complex part:
motion-only models with delays 1 to 15 frames, on the samples of each repeat (mirror-balanced and split
by location, region mean > 5).

In [ ]:
D2 = pd.DataFrame(delays).groupby("delay").auc_motion.agg(["mean", "std"])
print(D2.round(4).T.to_string())
sd1 = D2.loc[1, "std"]
print("no delay above delay 1 by more than its SD:", bool((D2["mean"].iloc[1:] <= D2.loc[1, "mean"] + sd1).all()),
      "; delay 15 below delay 1 by more than the SD:", bool(D2.loc[15, "mean"] < D2.loc[1, "mean"] - sd1),
      "; delay 1 the highest:", bool(D2["mean"].idxmax() == 1))

## Summary

| Claim | Outcome |
|---|---|
| fig5e-cue-betas | not reproduced under every file set and onset: with smoke-1a (assumed) it fails under every reading with the SI onset; with the code's smoke-2a files it fails with per-trajectory z-scoring; with the dataset's turn start and smoke-1a it holds at 160 to 200 degrees as stored or pooled and fails per trajectory and at 150 to 210 |
| fig5f-cue-dominance | partially reproduced, assumed (smoke-1a files and the SI onset): signs hold in 4 of 6 readings, values never both within tolerance; both within tolerance with smoke-2a at 150 to 210, or with smoke-1a and the dataset's turn start, so the outcome also rests on the onset asked of a reader |
| figs5c-turn-angles | reproduced, assumed (turn bounded by the 20% crossings); about -21 and +21 degrees over the files' events and about -26 and +27 over the half-height run, so a change to not reproduced is proposed |
| fig2c-complex-auc | partially reproduced, assumed (mirror-balanced samples): ordering and sum hold, values do not; not reproduced under the samplings the claims file names |
| fig2e-complex-weights | partially reproduced: ordering holds, gradient weight too large |
| figs1fg-resolution-robustness | not reproduced: motion r 0.73 to 0.78 at 3 px and 4 frames under every reading; gradient r 0.61 at 12 px (0.81 with smoothed frames) |
| methods-intensity-right-bias | blocked: metadata_insufficient; holds if -1 is a right turn, fails if +1 is |
| figs1-antisymmetry-index, figs1-temporal-statistics | blocked (smooth plume); the freestream and complex parts hold |
| the 13 other claims on the smooth plume, the networks and the agents | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The Fig. 5 analysis is a port of the paper's code (its
filters, smoothing, frame matching and models), run under the readings the claims file names, with
the SI's turn onset as the main reading and the smoke-1a files as the complex plume (assumed). The port
was not run beside the code here; its closest readings come within 0.1 SE of the code's published
tables. The Fig. 2 samples are drawn by a procedure the paper does not give in full (mirror-balanced,
assumed), with 10 repeats in place of 100. The motion cue of Fig. S1G is not defined in the paper;
three forms are run, with and without smoothing. The Fig. S1 parts use the dandiset's deposited
freestream maps and points, with masks from every 25th frame.

**What was found about the dataset.** The code's published Fig. 5 tables are matched most closely, within
0.1 SE and 0.001 in log-likelihood per turn, by the smoke-1a files with the dataset's turn start and the
SI's heading window, and closely by the smoke-1a files with the SI onset (0.45 SE), not by the smoke-2a
files and the 150 to 210 window the code names, which give a complex motion coefficient of -0.012
against +0.044. The two complex sets give coefficients of opposite sign; their intensity layouts are
not mirrored relative to each other, and the mean gradient cue of neither follows its mean intensity
profile, so a sign convention differing between them is not ruled out. The files do not say which
complex set is the paper's plume, which turn direction is left or right, or how the heading's axes are
oriented (theta is atan2(-vy, vx)). The turn events start a median of 4 frames (67 ms) after the SI's
onset computed from the files' dtheta and stop a median of 4 frames before the 20% crossing on the
falling edge; they lie within the run at or above half the peak. Four smooth files have frames far
outside the arena. The files give no movie frame for a trajectory frame, so the odor signals cannot be
aligned in time with the deposited movie. The smooth plume, needed by 15 claims, is in another archive
whose downloads answer 401 without a token.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")
import resource
print("peak memory", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")